#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

In [3]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [4]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split ,cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [6]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [7]:
train_data = pd.read_csv('data01_train.csv')

In [8]:
test_data = pd.read_csv('data01_test.csv')

#### 2) 기본 정보 조회

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [10]:
train_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [11]:
train_data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [12]:
train_data.drop('subject', axis=1, inplace=True)

In [13]:
train_data.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING


In [14]:
sum(train_data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [15]:
static_activity = (train_data['Activity']=='LAYING') | (train_data['Activity']=='SITTING') | (train_data['Activity']=='STANDING')
train_data = train_data[static_activity]

In [16]:
static_activity = (test_data['Activity']=='LAYING') | (test_data['Activity']=='SITTING') | (test_data['Activity']=='STANDING')
test_data = test_data[static_activity]

In [17]:
X_train = train_data.drop('Activity', axis=1)
y_train = train_data.loc[:, 'Activity']

In [18]:
X_test = test_data.drop(['subject', 'Activity'], axis=1)
y_test = test_data.loc[:, 'Activity']

In [19]:
y_train.unique()

array(['STANDING', 'LAYING', 'SITTING'], dtype=object)

In [20]:
y_test.unique()

array(['SITTING', 'STANDING', 'LAYING'], dtype=object)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [21]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [22]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [23]:
acc_df = pd.DataFrame({'모델': [],
                       '논문_정확도(교차검증)': [],
                       '논문_정확도(테스트)': [],
                       '논문_정확도(fi_교차검증)': [],
                       '논문_정확도(fi_테스트)': []
                      })
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)


In [24]:
kappa_df = pd.DataFrame({'모델': [],
                       '논문_kappa(교차검증)': [],
                       '논문_kappa(테스트)': [],
                      })
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)


In [25]:
kappa_scorer = make_scorer(cohen_kappa_score)

### (1) 알고리즘1 : RandomForest

In [26]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9560917021164604
평균 카파  : 0.9341073344896322


In [27]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9447779111644657
[[291   1   0]
 [  0 234  20]
 [  0  25 262]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.90      0.92      0.91       254
    STANDING       0.93      0.91      0.92       287

    accuracy                           0.94       833
   macro avg       0.94      0.94      0.94       833
weighted avg       0.95      0.94      0.94       833

Cohen's Kappa Score: 0.9171


In [28]:
joblib.dump(rf_model, 'rf_model_paper.pkl')

['rf_model_paper.pkl']

In [29]:
new_row = pd.DataFrame([{'모델': 'RandomForest', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [30]:
new_row = pd.DataFrame([{'모델': 'RandomForest', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [31]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.956092,0.944778,NaN,NaN


In [32]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.934107,0.91705


### (2) 알고리즘2 : LogisticRegressor

In [33]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9703159617381484
평균 카파  : 0.9554534317719611


In [34]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9675870348139256
[[290   2   0]
 [  0 243  11]
 [  0  14 273]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.94      0.96      0.95       254
    STANDING       0.96      0.95      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9513


In [35]:
joblib.dump(lr_model, 'lr_model_paper.pkl')

['lr_model_paper.pkl']

In [36]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [37]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [38]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.956092,0.944778,NaN,NaN
1,LogisticRegression,0.970316,0.967587,NaN,NaN


In [39]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.934107,0.917050
1,LogisticRegression,0.955453,0.951309


### (3) 알고리즘3 : SVM

In [40]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9740263468927797
평균 카파  : 0.9610196172198933


In [41]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9759903961584634
[[292   0   0]
 [  0 244  10]
 [  0  10 277]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.96      0.96      0.96       254
    STANDING       0.97      0.97      0.97       287

    accuracy                           0.98       833
   macro avg       0.98      0.98      0.98       833
weighted avg       0.98      0.98      0.98       833

Cohen's Kappa Score: 0.9639


In [42]:
joblib.dump(svm_model, 'svm_model_paper.pkl')

['svm_model_paper.pkl']

In [43]:
new_row = pd.DataFrame([{'모델': 'SVM', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [44]:
new_row = pd.DataFrame([{'모델': 'SVM', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [45]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.956092,0.944778,NaN,NaN
1,LogisticRegression,0.970316,0.967587,NaN,NaN
2,SVM,0.974026,0.975990,NaN,NaN


In [46]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.934107,0.917050
1,LogisticRegression,0.955453,0.951309
2,SVM,0.961020,0.963919


### (4) 알고리즘4 : DecisionTreeClassifier

In [47]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9350677812815519
평균 카파  : 0.9025685765609228


In [48]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9207683073229291
[[290   2   0]
 [  0 234  20]
 [  0  44 243]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.84      0.92      0.88       254
    STANDING       0.92      0.85      0.88       287

    accuracy                           0.92       833
   macro avg       0.92      0.92      0.92       833
weighted avg       0.92      0.92      0.92       833

Cohen's Kappa Score: 0.8812


In [49]:
joblib.dump(dt_model, 'dt_model_paper.pkl')

['dt_model_paper.pkl']

In [50]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', '논문_정확도(교차검증)': np.mean(scores['test_accuracy']), '논문_정확도(테스트)': accuracy_score(y_test, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [51]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', '논문_kappa(교차검증)': np.mean(scores['test_kappa']), '논문_kappa(테스트)': cohen_kappa_score(y_test, y_pred)}])
kappa_df = pd.concat([kappa_df, new_row], ignore_index=True)

In [52]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.956092,0.944778,NaN,NaN
1,LogisticRegression,0.970316,0.967587,NaN,NaN
2,SVM,0.974026,0.975990,NaN,NaN
3,DecisionTreeClassifier,0.935068,0.920768,NaN,NaN


In [53]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.934107,0.917050
1,LogisticRegression,0.955453,0.951309
2,SVM,0.961020,0.963919
3,DecisionTreeClassifier,0.902569,0.881156


In [54]:
acc_df.to_csv('모델 정확도(논문 기준).csv', index=False, encoding='utf-8')

In [55]:
kappa_df.to_csv('모델 정확도(논문 기준).csv', index=False, encoding='utf-8')

### (5) 알고리즘5 : XGBoost

In [56]:
# le = LabelEncoder()
# y_xgb = le.fit_transform(y)

In [57]:
# X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X, y_xgb, test_size=0.3, random_state=42)

In [58]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train_xgb, y_train_xgb)

# y_pred = XGB_model.predict(X_val_xgb)
# y_pred = le.inverse_transform(y_pred)

# y_val = le.inverse_transform(y_val_xgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [59]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_val, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [60]:
# acc_df

### (5) 알고리즘5 : LightGBM

In [61]:
# X_lgb = X.copy()
# X_lgb.columns = [f'feature_{i}' for i in range(X.shape[1])]

In [62]:
# le = LabelEncoder()
# y_lgb = le.fit_transform(y)

In [63]:
# X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_lgb, y_lgb, test_size=0.3, random_state=42)

In [64]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_val_lgb)
# y_pred = le.inverse_transform(y_pred)

# y_val= le.inverse_transform(y_val_lgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [65]:
# y_pred = model.predict(x_val)
# y_pred = np.argmax(y_pred, axis=1)

In [66]:
# y_val = le.inverse_transform(y_val)
# y_pred = le.inverse_transform(y_pred)
# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

# =====특성 중요도=======================


## 방법1. 랜덤포레스트에서 추출한 feature_importance_ 사용

In [67]:
acc_df = pd.read_csv('모델 정확도(논문 기준).csv')
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트)
0,RandomForest,0.934107,0.917050
1,LogisticRegression,0.955453,0.951309
2,SVM,0.961020,0.963919
3,DecisionTreeClassifier,0.902569,0.881156


In [68]:
import joblib
rf_model = joblib.load('rf_model_paper.pkl')

In [69]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,tGravityAcc-mean()-X,0.075933
1,"angle(X,gravityMean)",0.073077
2,tGravityAcc-mean()-Y,0.072572
3,tGravityAcc-max()-Y,0.060073
4,"angle(Z,gravityMean)",0.059058
...,...,...
556,fBodyAccMag-skewness(),0.000000
557,fBodyBodyAccJerkMag-iqr(),0.000000
558,"tBodyAcc-arCoeff()-Z,2",0.000000
559,tBodyAcc-iqr()-Y,0.000000


In [70]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [71]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [72]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 151


In [73]:
selected_cum_importance = RFmodel_fi_df.loc[:len(selected_features), 'feature_importance'].sum()
print(f"상위 {len(selected_features)+1}개 변수의 누적 중요도: {selected_cum_importance:.4f}")

상위 152개 변수의 누적 중요도: 0.9501


In [74]:
X_train_fi = train_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]
y_train_fi = train_data.loc[:, 'Activity']

In [75]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [76]:
X_test_fi = test_data[RFmodel_fi_df['feature_name'][:len(selected_features)+1]]

### 랜덤 포레스트

In [77]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9492882128040347
평균 카파  : 0.9239087794451075


In [78]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train_fi, y_train_fi)
y_pred = rf_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9687875150060024
[[290   2   0]
 [  0 242  12]
 [  0  12 275]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.95      0.95      0.95       254
    STANDING       0.96      0.96      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9531


In [79]:
rf_model = joblib.dump(rf_model, 'rf_model_paper_fi.pkl')

In [80]:
acc_df.loc[0, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [81]:
kappa_df.loc[0, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [82]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788
1,LogisticRegression,0.955453,0.951309,NaN,NaN
2,SVM,0.961020,0.963919,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


In [83]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103
1,LogisticRegression,0.955453,0.951309,NaN,NaN
2,SVM,0.961020,0.963919,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


### LinearRegression

In [84]:
# 랜덤 포레스트에서 추출한 중요도 상위 특성 활용
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9659868600494782
평균 카파  : 0.9489596241418561


In [85]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_fi, y_train_fi)
y_pred = lr_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9699879951980792
[[290   2   0]
 [  0 245   9]
 [  0  14 273]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.94      0.96      0.95       254
    STANDING       0.97      0.95      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9549


In [86]:
joblib.dump(lr_model, 'lr_model_paper_fi.pkl')

['lr_model_paper_fi.pkl']

In [87]:
acc_df.loc[1, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [88]:
kappa_df.loc[1, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [89]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988
2,SVM,0.961020,0.963919,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


In [90]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922
2,SVM,0.961020,0.963919,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


### SVM

In [91]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.963824462510946
평균 카파  : 0.9457164208124548


In [92]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_fi, y_train_fi)

y_pred = svm_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9699879951980792
[[292   0   0]
 [  0 246   8]
 [  0  17 270]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.94      0.97      0.95       254
    STANDING       0.97      0.94      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9549


In [93]:
joblib.dump(svm_model, 'svm_model_paper_fi.pkl')

['svm_model_paper_fi.pkl']

In [94]:
acc_df.loc[2, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [95]:
kappa_df.loc[2, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [96]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988
2,SVM,0.961020,0.963919,0.963824,0.969988
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


In [97]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922
2,SVM,0.961020,0.963919,0.945716,0.954928
3,DecisionTreeClassifier,0.902569,0.881156,NaN,NaN


### DecisionTreeClassifier

In [98]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_fi, y_train_fi, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9359951383140095
평균 카파  : 0.9039614443517123


In [99]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_fi, y_train_fi)

y_pred = dt_model.predict(X_test_fi)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9207683073229291
[[290   2   0]
 [  0 234  20]
 [  0  44 243]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.84      0.92      0.88       254
    STANDING       0.92      0.85      0.88       287

    accuracy                           0.92       833
   macro avg       0.92      0.92      0.92       833
weighted avg       0.92      0.92      0.92       833

Cohen's Kappa Score: 0.8812


In [100]:
joblib.dump(dt_model, 'dt_model_paper_fi.pkl')

['dt_model_paper_fi.pkl']

In [101]:
acc_df.loc[3, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(fi_테스트)'] = accuracy_score(y_test, y_pred)

In [102]:
kappa_df.loc[3, '논문_kappa(fi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(fi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [103]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988
2,SVM,0.961020,0.963919,0.963824,0.969988
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768


In [104]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922
2,SVM,0.961020,0.963919,0.945716,0.954928
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156


In [105]:
acc_df.to_csv('모델 정확도(논문 기준_fi).csv', index=False, encoding='utf-8-sig')

In [106]:
kappa_df.to_csv('모델 kappa(논문 기준_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [107]:
acc_df = pd.read_csv('모델 정확도(논문 기준_fi).csv')
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988
2,SVM,0.961020,0.963919,0.963824,0.969988
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768


In [108]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 76
누적 설명력: 0.9508246619476548


In [109]:
X_test_pca = pca.transform(X_test)

### 랜덤포레스트

In [110]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.894865083428637
평균 카파  : 0.8420844440922458


In [111]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train_pca, y_train)
y_pred = rf_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8931572629051621
[[291   1   0]
 [  5 192  57]
 [  0  26 261]]
              precision    recall  f1-score   support

      LAYING       0.98      1.00      0.99       292
     SITTING       0.88      0.76      0.81       254
    STANDING       0.82      0.91      0.86       287

    accuracy                           0.89       833
   macro avg       0.89      0.89      0.89       833
weighted avg       0.89      0.89      0.89       833

Cohen's Kappa Score: 0.8390


In [112]:
acc_df.loc[0, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [113]:
kappa_df.loc[0, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [114]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,NaN,NaN
2,SVM,0.961020,0.963919,0.963824,0.969988,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,NaN,NaN


In [115]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.83903
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,NaN,NaN
2,SVM,0.961020,0.963919,0.945716,0.954928,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,NaN,NaN


### 로지스틱회귀

In [116]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', max_iter=5000, random_state=42) # 최적점에 도달 못하여 max_iter=5000로 설정
scores = cross_validate(lr_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9641331030093644
평균 카파  : 0.9461693295357158


In [117]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_pca, y_train)
y_pred = lr_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9543817527010804
[[292   0   0]
 [  0 237  17]
 [  0  21 266]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.92      0.93      0.93       254
    STANDING       0.94      0.93      0.93       287

    accuracy                           0.95       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9315


In [118]:
acc_df.loc[1, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [119]:
kappa_df.loc[1, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [120]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382
2,SVM,0.961020,0.963919,0.963824,0.969988,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,NaN,NaN


In [121]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466
2,SVM,0.961020,0.963919,0.945716,0.954928,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,NaN,NaN


### SVM

In [122]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9632047889521058
평균 카파  : 0.9447799157152943


In [123]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9543817527010804
[[292   0   0]
 [  0 237  17]
 [  0  21 266]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.92      0.93      0.93       254
    STANDING       0.94      0.93      0.93       287

    accuracy                           0.95       833
   macro avg       0.95      0.95      0.95       833
weighted avg       0.95      0.95      0.95       833

Cohen's Kappa Score: 0.9315


In [124]:
acc_df.loc[2, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [125]:
kappa_df.loc[2, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [126]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,NaN,NaN


In [127]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,NaN,NaN


### 결정 트리

In [128]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_pca, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.8320866490255096
평균 카파  : 0.7479936449507036


In [129]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_pca, y_train)

y_pred = dt_model.predict(X_test_pca)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.8187274909963985
[[279   4   9]
 [ 15 177  62]
 [  0  61 226]]
              precision    recall  f1-score   support

      LAYING       0.95      0.96      0.95       292
     SITTING       0.73      0.70      0.71       254
    STANDING       0.76      0.79      0.77       287

    accuracy                           0.82       833
   macro avg       0.81      0.81      0.81       833
weighted avg       0.82      0.82      0.82       833

Cohen's Kappa Score: 0.7273


In [130]:
acc_df.loc[3, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(pca_테스트)'] = accuracy_score(y_test, y_pred)

In [131]:
kappa_df.loc[3, '논문_kappa(pca_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(pca_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [132]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727


In [133]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,0.747994,0.727349


In [134]:
acc_df.to_csv('모델 정확도(논문 기준_fi_pca).csv', index=False, encoding='utf-8-sig')

In [135]:
kappa_df.to_csv('모델 kappa(논문 기준_fi_pca).csv', index=False, encoding='utf-8-sig')

## 방법3: permutation_importance()

In [136]:
acc_df = pd.read_csv('모델 정확도(논문 기준_fi_pca).csv')
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727


In [137]:
rf_model = joblib.load('rf_model_paper.pkl')
lr_model = joblib.load('lr_model_paper.pkl')
svm_model = joblib.load('svm_model_paper.pkl')
dt_model = joblib.load('dt_model_paper.pkl')

### 랜덤포레스트

In [138]:
model = rf_model

# permutation importance 계산
result_rf = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [139]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result_rf.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 179


In [140]:
importances_df.to_csv('rf_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [141]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9598025657834922
평균 카파  : 0.9396663588994821


In [142]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train_sel, y_train)
y_pred = rf_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.957983193277311
[[291   1   0]
 [  0 234  20]
 [  0  14 273]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.94      0.92      0.93       254
    STANDING       0.93      0.95      0.94       287

    accuracy                           0.96       833
   macro avg       0.96      0.96      0.96       833
weighted avg       0.96      0.96      0.96       833

Cohen's Kappa Score: 0.9368


In [143]:
acc_df.loc[0, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[0, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [144]:
kappa_df.loc[0, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[0, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [145]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157,0.959803,0.957983
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382,NaN,NaN
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727,NaN,NaN


In [146]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030,0.939666,0.936837
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466,NaN,NaN
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,0.747994,0.727349,NaN,NaN


### 로지스틱회귀

In [147]:
model = lr_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [148]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 42


In [149]:
importances_df.to_csv('lr_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [150]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', max_iter=5000, random_state=42)
scores = cross_validate(lr_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9721735468774673
평균 카파  : 0.9582389869260215


In [151]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_sel, y_train)
y_pred = lr_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9735894357743097
[[290   2   0]
 [  0 244  10]
 [  0  10 277]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.95      0.96      0.96       254
    STANDING       0.97      0.97      0.97       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9603


In [152]:
acc_df.loc[1, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[1, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [153]:
kappa_df.loc[1, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[1, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [154]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157,0.959803,0.957983
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382,0.972174,0.973589
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727,NaN,NaN


In [155]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030,0.939666,0.936837
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466,0.958239,0.960318
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466,NaN,NaN
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,0.747994,0.727349,NaN,NaN


### SVM

In [156]:
model = svm_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1) # n_repeats=30 -> 너무 오래 걸림...30분이 넘게 걸려 결국 포기.  n_repeats=10 으로 실행
# 너무 오래 걸려서 안되면.... permutation_importance()를 랜덤포레스트로 하고 훈련을 svm으로 시키는 것을 고려할 것임.

In [157]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 127


In [158]:
importances_df.to_csv('svm_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [159]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9768108105521554
평균 카파  : 0.9651966993463841


In [160]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_sel, y_train)

y_pred = svm_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9711884753901561
[[292   0   0]
 [  0 242  12]
 [  0  12 275]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       292
     SITTING       0.95      0.95      0.95       254
    STANDING       0.96      0.96      0.96       287

    accuracy                           0.97       833
   macro avg       0.97      0.97      0.97       833
weighted avg       0.97      0.97      0.97       833

Cohen's Kappa Score: 0.9567


In [161]:
acc_df.loc[2, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[2, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [162]:
kappa_df.loc[2, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[2, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [163]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157,0.959803,0.957983
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382,0.972174,0.973589
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382,0.976811,0.971188
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727,NaN,NaN


In [164]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030,0.939666,0.936837
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466,0.958239,0.960318
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466,0.965197,0.956703
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,0.747994,0.727349,NaN,NaN


### 결정트리

In [165]:
model = dt_model

# permutation importance 계산
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1)

In [166]:
# 중요도 정리 + 누적 중요도 계산
importances_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 중요도가 양수인 피처만 선택
importances_df = importances_df[importances_df['importance'] > 0]

print(f"선택된 피처 수: {len(importances_df)}")

# 피처 이름 추출
selected_features = importances_df['feature']

# 선택된 피처로 데이터 재구성
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

선택된 피처 수: 7


In [167]:
importances_df.to_csv('dt_permutation_importance_DataFrame(논문 기준).csv', index=False, encoding='utf-8-sig')

In [168]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_sel, y_train, cv=5, scoring={'accuracy': 'accuracy', 'kappa': kappa_scorer})
print(f"평균 정확도:", np.mean(scores['test_accuracy']))
print(f"평균 카파  :", np.mean(scores['test_kappa']))

평균 정확도: 0.9449605466525665
평균 카파  : 0.9174268463080896


In [169]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_sel, y_train)

y_pred = dt_model.predict(X_test_sel)

print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 0.9207683073229291
[[290   2   0]
 [  0 234  20]
 [  0  44 243]]
              precision    recall  f1-score   support

      LAYING       1.00      0.99      1.00       292
     SITTING       0.84      0.92      0.88       254
    STANDING       0.92      0.85      0.88       287

    accuracy                           0.92       833
   macro avg       0.92      0.92      0.92       833
weighted avg       0.92      0.92      0.92       833

Cohen's Kappa Score: 0.8812


In [170]:
acc_df.loc[3, '논문_정확도(pi_교차검증)'] = np.mean(scores['test_accuracy'])
acc_df.loc[3, '논문_정확도(pi_테스트)'] = accuracy_score(y_test, y_pred)

In [171]:
kappa_df.loc[3, '논문_kappa(pi_교차검증)'] = np.mean(scores['test_kappa'])
kappa_df.loc[3, '논문_kappa(pi_테스트)'] = cohen_kappa_score(y_test, y_pred)

In [172]:
acc_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트),논문_정확도(pi_교차검증),논문_정확도(pi_테스트)
0,RandomForest,0.934107,0.917050,0.949288,0.968788,0.894865,0.893157,0.959803,0.957983
1,LogisticRegression,0.955453,0.951309,0.965987,0.969988,0.964133,0.954382,0.972174,0.973589
2,SVM,0.961020,0.963919,0.963824,0.969988,0.963205,0.954382,0.976811,0.971188
3,DecisionTreeClassifier,0.902569,0.881156,0.935995,0.920768,0.832087,0.818727,0.944961,0.920768


In [173]:
kappa_df

,모델,논문_kappa(교차검증),논문_kappa(테스트),논문_kappa(fi_교차검증),논문_kappa(fi_테스트),논문_kappa(pca_교차검증),논문_kappa(pca_테스트),논문_kappa(pi_교차검증),논문_kappa(pi_테스트)
0,RandomForest,0.934107,0.917050,0.923909,0.953103,0.842084,0.839030,0.939666,0.936837
1,LogisticRegression,0.955453,0.951309,0.948960,0.954922,0.946169,0.931466,0.958239,0.960318
2,SVM,0.961020,0.963919,0.945716,0.954928,0.944780,0.931466,0.965197,0.956703
3,DecisionTreeClassifier,0.902569,0.881156,0.903961,0.881156,0.747994,0.727349,0.917427,0.881156


In [174]:
acc_df.to_csv('모델 정확도(논문 기준_fi_pca_pi).csv', index=False, encoding='utf-8-sig')

In [175]:
kappa_df.to_csv('모델 kappa(논문 기준_fi_pca_pi).csv', index=False, encoding='utf-8-sig')